# ABC Inflation Forecasting: Beating SARB Forecasts

This notebook demonstrates how to use Approximate Bayesian Computation (ABC) for inflation forecasting and compare the results against South African Reserve Bank (SARB) forecasts.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sys
sys.path.insert(0, '../src')

from abc_inflation.models import ABCInflationModel, ARIMAModel, SARIMAModel
from abc_inflation.utils import (
    forecast_metrics,
    compare_forecasts,
    generate_summary_statistics,
    create_train_test_split,
    plot_forecasts,
    plot_comparison,
    plot_residuals,
    plot_posterior_distributions,
    create_metrics_table
)

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 2. Load and Prepare Data

In this example, we'll simulate inflation data. In practice, you would load actual South African inflation data here.

In [ ]:
# Simulate inflation data (replace with actual data loading)
def simulate_inflation_data(n_periods=120, seed=42):
    """
    Simulate realistic inflation data with trend, seasonality, and noise.
    """
    np.random.seed(seed)
    
    # Time index
    time = np.arange(n_periods)
    
    # Trend component
    trend = 4.5 + 0.5 * np.sin(time / 20)
    
    # Seasonal component (annual)
    seasonal = 0.8 * np.sin(2 * np.pi * time / 12)
    
    # AR(1) component
    ar_component = np.zeros(n_periods)
    ar_component[0] = np.random.normal(0, 0.5)
    for t in range(1, n_periods):
        ar_component[t] = 0.7 * ar_component[t-1] + np.random.normal(0, 0.5)
    
    # Combine components
    inflation = trend + seasonal + ar_component
    
    # Create DataFrame
    dates = pd.date_range(start='2014-01-01', periods=n_periods, freq='M')
    df = pd.DataFrame({'inflation': inflation}, index=dates)
    
    return df

# Generate data
df = simulate_inflation_data(n_periods=120)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"\nSummary statistics:")
print(df.describe())

In [ ]:
# Visualize the data
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['inflation'], 'b-', linewidth=2)
plt.xlabel('Time', fontsize=12)
plt.ylabel('Inflation Rate (%)', fontsize=12)
plt.title('Simulated Inflation Data', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Split Data into Train and Test Sets

In [ ]:
# Use last 12 months for testing
test_size = 12
train_data = df['inflation'].values[:-test_size]
test_data = df['inflation'].values[-test_size:]
train_dates = df.index[:-test_size]
test_dates = df.index[-test_size:]

print(f"Training data: {len(train_data)} observations")
print(f"Testing data: {len(test_data)} observations")

## 4. Fit Baseline Models

### 4.1 ARIMA Model

In [ ]:
# Fit ARIMA model
arima_model = ARIMAModel(order=(2, 1, 2))
arima_model.fit(train_data)

# Generate forecasts
arima_forecast, arima_lower, arima_upper = arima_model.forecast(test_size)

print("ARIMA Model Summary:")
print(arima_model.get_params())

### 4.2 SARIMA Model

In [ ]:
# Fit SARIMA model
sarima_model = SARIMAModel(order=(1, 1, 1), seasonal_order=(1, 0, 1, 12))
sarima_model.fit(train_data)

# Generate forecasts
sarima_forecast, sarima_lower, sarima_upper = sarima_model.forecast(test_size)

print("SARIMA Model Summary:")
print(sarima_model.get_params())

## 5. Fit ABC Model

In [ ]:
# Define prior distributions for ABC
prior_distributions = {
    'phi': stats.uniform(0.3, 0.6),      # AR coefficient: uniform(0.3, 0.9)
    'sigma': stats.uniform(0.1, 1.0),    # Noise std: uniform(0.1, 1.1)
    'mu': stats.norm(4.5, 1.0),          # Mean: normal(4.5, 1.0)
}

# Initialize ABC model
abc_model = ABCInflationModel(
    prior_distributions=prior_distributions,
    summary_stats_fn=generate_summary_statistics,
    distance_metric='euclidean',
    n_particles=500,
    epsilon_schedule=[2.0, 1.0, 0.5, 0.25]
)

print("ABC Model initialized with:")
print(f"  - {abc_model.n_particles} particles")
print(f"  - Epsilon schedule: {abc_model.epsilon_schedule}")
print(f"  - Distance metric: {abc_model.distance_metric}")

In [ ]:
# Fit ABC model using SMC
print("Fitting ABC model (this may take a few minutes)...")
abc_model.fit(train_data, method='smc', verbose=True)

print("\nABC fitting complete!")

In [ ]:
# View posterior summary
posterior_summary = abc_model.get_posterior_summary()
print("\nPosterior Summary:")
for param, stats_dict in posterior_summary.items():
    print(f"\n{param}:")
    print(f"  Mean: {stats_dict['mean']:.4f}")
    print(f"  Std: {stats_dict['std']:.4f}")
    print(f"  95% CI: [{stats_dict['q025']:.4f}, {stats_dict['q975']:.4f}]")

In [ ]:
# Plot posterior distributions
plot_posterior_distributions(abc_model.posterior_samples)

In [ ]:
# Generate ABC forecasts
abc_forecast, abc_lower, abc_upper = abc_model.forecast(
    n_steps=test_size,
    n_simulations=200,
    initial_conditions=train_data[-10:]
)

print(f"ABC forecast generated for {test_size} steps")

## 6. Compare Forecasts

In [ ]:
# Visualize all forecasts
forecasts_dict = {
    'ARIMA': (arima_forecast, arima_lower, arima_upper),
    'SARIMA': (sarima_forecast, sarima_lower, sarima_upper),
    'ABC': (abc_forecast, abc_lower, abc_upper),
}

# Create combined date index
all_dates = pd.concat([train_dates.to_series(), test_dates.to_series()])
all_data = np.concatenate([train_data, test_data])

plot_forecasts(
    actual=all_data,
    forecasts=forecasts_dict,
    dates=all_dates.index,
    title='Inflation Forecasts Comparison',
    ylabel='Inflation Rate (%)'
)

## 7. Evaluate Forecast Performance

In [ ]:
# Calculate metrics for all models
metrics = compare_forecasts(
    actual=test_data,
    forecasts_dict={
        'ARIMA': arima_forecast,
        'SARIMA': sarima_forecast,
        'ABC': abc_forecast,
    }
)

# Create metrics table
metrics_df = create_metrics_table(metrics)
print("\nForecast Performance Metrics:")
print(metrics_df[['MAE', 'RMSE', 'MAPE', 'R2']])

In [ ]:
# Visualize metric comparison
plot_comparison(
    metrics_dict=metrics,
    metric_names=['RMSE', 'MAE', 'MAPE'],
    title='Forecast Accuracy Comparison'
)

In [ ]:
# Calculate coverage probabilities
from abc_inflation.utils.metrics import coverage_probability

coverage_results = {}
for model_name, (mean, lower, upper) in forecasts_dict.items():
    coverage = coverage_probability(test_data, lower, upper, nominal_coverage=0.95)
    coverage_results[model_name] = coverage

print("\n95% Prediction Interval Coverage:")
for model, coverage in coverage_results.items():
    print(f"{model}: {coverage:.2%}")

## 8. Compare with SARB Forecasts

In practice, you would load actual SARB forecasts here and compare them with the ABC model.

In [ ]:
# Simulate SARB forecasts for demonstration
# In practice, load actual SARB forecasts from data file
sarb_forecast = test_data + np.random.normal(0, 0.5, len(test_data))
sarb_lower = sarb_forecast - 1.0
sarb_upper = sarb_forecast + 1.0

# Add to comparison
forecasts_with_sarb = forecasts_dict.copy()
forecasts_with_sarb['SARB'] = (sarb_forecast, sarb_lower, sarb_upper)

# Calculate metrics including SARB
metrics_with_sarb = compare_forecasts(
    actual=test_data,
    forecasts_dict={
        'ARIMA': arima_forecast,
        'SARIMA': sarima_forecast,
        'ABC': abc_forecast,
        'SARB': sarb_forecast,
    }
)

# Display results
metrics_df_sarb = create_metrics_table(metrics_with_sarb)
print("\nForecast Performance vs SARB:")
print(metrics_df_sarb[['MAE', 'RMSE', 'MAPE', 'R2']])

In [ ]:
# Visualize comparison with SARB
plot_comparison(
    metrics_dict=metrics_with_sarb,
    metric_names=['RMSE', 'MAE', 'MAPE'],
    title='Forecast Accuracy: ABC vs SARB vs Baselines'
)

## 9. Statistical Comparison

Perform Diebold-Mariano test to check if ABC forecasts are significantly better than SARB.

In [ ]:
from abc_inflation.utils.metrics import diebold_mariano_test

# Calculate forecast errors
abc_errors = test_data - abc_forecast
sarb_errors = test_data - sarb_forecast
arima_errors = test_data - arima_forecast
sarima_errors = test_data - sarima_forecast

# Compare ABC vs SARB
dm_abc_sarb = diebold_mariano_test(abc_errors, sarb_errors)
print("\nDiebold-Mariano Test: ABC vs SARB")
print(f"  DM Statistic: {dm_abc_sarb['DM_statistic']:.4f}")
print(f"  P-value: {dm_abc_sarb['p_value']:.4f}")
print(f"  Significant difference: {dm_abc_sarb['significant']}")

# Compare ABC vs ARIMA
dm_abc_arima = diebold_mariano_test(abc_errors, arima_errors)
print("\nDiebold-Mariano Test: ABC vs ARIMA")
print(f"  DM Statistic: {dm_abc_arima['DM_statistic']:.4f}")
print(f"  P-value: {dm_abc_arima['p_value']:.4f}")
print(f"  Significant difference: {dm_abc_arima['significant']}")

## 10. Residual Analysis

In [ ]:
# Plot residuals for ABC model
plot_residuals(test_data, abc_forecast, model_name='ABC')

## 11. Conclusions

This notebook demonstrates:

1. **ABC Framework Implementation**: A complete implementation of Approximate Bayesian Computation with Sequential Monte Carlo for inflation forecasting.

2. **Model Comparison**: Comparison of ABC against traditional methods (ARIMA, SARIMA) and SARB forecasts.

3. **Performance Metrics**: Comprehensive evaluation using multiple metrics (MAE, RMSE, MAPE, coverage probability).

4. **Statistical Testing**: Formal comparison using Diebold-Mariano test.

### Key Advantages of ABC Approach:

- **Flexibility**: Can incorporate complex model structures without requiring likelihood functions
- **Uncertainty Quantification**: Provides full posterior distributions for parameters
- **Robust Forecasts**: Accounts for parameter uncertainty in predictions
- **Interpretability**: Clear understanding of model assumptions through prior specifications

### Next Steps:

1. Load actual South African inflation data
2. Incorporate SARB official forecasts
3. Extend ABC model to include macroeconomic indicators
4. Implement more sophisticated model structures (e.g., stochastic volatility)
5. Perform rolling window validation
6. Generate policy-relevant forecast scenarios